# Turtle Trading Strategy, Backtested on Crypto
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nikshev/unstuck/blob/main/episodes/q01-turtle-trading/q01_turtle_trading.ipynb)

The complete Turtle rules as written in Curtis Faith, *The Original Turtle Trading Rules* (2003):
intraday Donchian breakouts, the System 1 filter and the 55-day failsafe, N-based units,
pyramiding every ½ N, a 2N stop per unit, unit limits for correlated markets and the shrinking
notional account. On Bitcoin and Ethereum perpetual futures, January 2020 to October 2026, long and
short, with fees, funding and liquidation. Education, not financial advice.

What we cannot copy exactly: the Turtles entered **at** the breakout price with an order waiting in the
market. freqtrade has no stop-entry order, so we watch 5-minute candles and enter at the next one —
within five minutes of the breakout.

## 1. Install
freqtrade is the backtester; we call only a few of its methods and explain each one.

In [ ]:
!pip install -q freqtrade==2026.10

## 2. The data loader from episode q00
Raw Binance files, each checked against its SHA-256 hash — the code is built line by line in episode q00.

In [ ]:
!curl -fsSLO https://raw.githubusercontent.com/nikshev/unstuck/main/episodes/quant-lib/binance_vision.py
from binance_vision import save_for_freqtrade, klines, DATA
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

## 3. Bitcoin and Ethereum, 2020–2026
Daily candles for the channels and N, 5-minute candles to watch for the breakout (15-minute ones for a check at the end), plus mark price and funding. Uses `save_for_freqtrade`, `klines`, `DATA` (section 2).

In [ ]:
for symbol in ["BTCUSDT", "ETHUSDT"]:
    save_for_freqtrade(symbol, "1d", "2020-01", "2026-10")
    pair = symbol.replace("USDT", "_USDT_USDT")
    for tf in ["5m", "15m"]:
        candles = klines(symbol, tf, "2020-01", "2026-10")
        candles.to_feather(DATA / f"{pair}-{tf}-futures.feather")
daily_file = DATA / "{}_USDT_USDT-1d-futures.feather"
btc = pd.read_feather(str(daily_file).format("BTC")).set_index("date")
eth = pd.read_feather(str(daily_file).format("ETH")).set_index("date")
five = pd.read_feather(DATA / "BTC_USDT_USDT-5m-futures.feather")
print(len(btc), "days;", len(five), "five-minute candles per coin")

## 4. The breakout: a 20-day Donchian channel
Upper line = the highest high of the **preceding** 20 days, lower line = the lowest low. Faith: a breakout is the price exceeding it during the day. Uses `btc` (section 3).

In [ ]:
# shift(): today is not part of its own channel - no peeking
upper = btc["high"].rolling(20).max().shift()
lower = btc["low"].rolling(20).min().shift()
part = btc.loc["2020-08-01":"2021-04-30"]          # a strong trend to look at
plt.figure(figsize=(11, 5))
plt.plot(part.index, part["close"], label="BTC close")
plt.plot(part.index, upper[part.index], label="20-day high")
plt.plot(part.index, lower[part.index], label="20-day low")
plt.yscale("log"); plt.legend(); plt.title("Donchian channel, 20 days")
plt.show()
first = part[part["high"] > upper[part.index]].index[0]   # first breakout
print("first day the price exceeded the 20-day high:", first.date(),
      "level", upper[first])

## 5. N — the Turtles' measure of volatility
True range $TR = \max(H-L,\ |H-C_{prev}|,\ |L-C_{prev}|)$; $N_t = \dfrac{19 N_{t-1} + TR_t}{20}$ (Faith, "Volatility – The Meaning of N"). TR = true range. Uses `btc` (3), `first` (4).

In [ ]:
prev = btc["close"].shift()                         # yesterday's close
tr = pd.concat([btc["high"] - btc["low"],           # today's range
                (btc["high"] - prev).abs(),         # gap up from yesterday
                (btc["low"] - prev).abs()],         # gap down from yesterday
               axis=1).max(axis=1)
N = tr.ewm(alpha=1 / 20, adjust=False).mean()       # (19 x old N + TR) / 20
print("N on", first.date(), "=", round(N[first], 2), "USDT per BTC per day")

## 6. One unit risks 1% of the account per N
$\text{unit} = \dfrac{1\% \times \text{account}}{N}$ coins, so a move of one N against one unit costs 1% of the account. Uses `N`, `first` (5), `btc` (3).

In [ ]:
account = 10_000
unit = 0.01 * account / N[first]                   # coins in one unit
print("unit:", round(unit, 3), "BTC  =", round(unit * btc.loc[first, "close"]),
      "USDT of Bitcoin")
print("a 1 N move costs:", round(unit * N[first], 2), "USDT")

## 7. Backtest settings
Futures, isolated margin, 10,000 USDT (Tether, a dollar stablecoin), both coins, the stake decided by the strategy (`unlimited`), decisions on 5-minute candles.

In [ ]:
config = {
    "trading_mode": "futures", "margin_mode": "isolated",
    "liquidation_buffer": 0.0,
    "stake_currency": "USDT",
    "stake_amount": "unlimited",              # the strategy sizes each unit
    "dry_run_wallet": 10000, "dry_run": True,
    "max_open_trades": 2,                     # one position per coin
    "timeframe": "5m", "fee": 0.0005,
    "entry_pricing": {"price_side": "same", "use_order_book": True,
                      "order_book_top": 1},
    "exit_pricing": {"price_side": "same", "use_order_book": True,
                     "order_book_top": 1},
    "exchange": {"name": "binance",
                 "pair_whitelist": ["BTC/USDT:USDT", "ETH/USDT:USDT"]},
    "pairlists": [{"method": "StaticPairList"}],
}
json.dump(config, open("config.json", "w"), indent=1)

## 8. Strategy file, part 1: the rule book
System 1: 20-day entry, 10-day exit, 55-day failsafe, the filter on. 1% per unit; 4 units per market; 6 in closely correlated markets (Faith's table — BTC and ETH, Bitcoin and Ethereum, are one such group). 3x leverage only locks margin.

In [ ]:
%%writefile user_data/strategies/Turtle.py
import numpy as np
import pandas as pd
from freqtrade.persistence import Trade
from freqtrade.strategy import IStrategy, informative, stoploss_from_absolute

class Turtle(IStrategy):
    # Faith, System 1: 20-day entry, 10-day exit, 55-day failsafe, filter on
    ENTRY, EXIT, FAILSAFE, FILTER = 20, 10, 55, True
    # 1% risk per unit; 3x only locks margin; Binance maintenance margin rate
    RISK, LEV, MMR = 0.01, 3, 0.004
    # Faith's limits: 4 units per market, 6 in correlated markets (BTC + ETH)
    MAX_UNITS, MAX_CORRELATED = 4, 6
    timeframe = "5m"                    # look for breakouts every 5 minutes
    can_short = True
    minimal_roi = {"0": 100}            # no profit target: only exits and stops
    stoploss = -0.99                    # compulsory; the real stop is custom
    use_custom_stoploss = True
    position_adjustment_enable = True   # lets us add units to a position
    startup_candle_count = 2400

## 9. Part 2: the daily numbers
`@informative("1d")` computes on daily candles and hands every 5-minute candle the values of the **last finished day** — no peeking. Same N as section 5; the Turtles got a unit-size sheet every Monday, so sizing uses Monday's N. `ENTRY`, `EXIT`, `FAILSAFE` from section 8.

In [ ]:
%%writefile -a user_data/strategies/Turtle.py

    # computed on daily candles; 5-minute rows get the last FINISHED day (_1d)
    @informative("1d")
    def populate_indicators_1d(self, df, metadata):
        prev = df["close"].shift()
        tr = pd.concat([df["high"] - df["low"],          # true range, as in
                        (df["high"] - prev).abs(),       # section 5
                        (df["low"] - prev).abs()], axis=1).max(axis=1)
        df["N"] = tr.ewm(alpha=1 / 20, adjust=False).mean()
        # Faith: unit sizes were set every Monday -> size with Monday's N
        df["N_sheet"] = df["N"].where(df["date"].dt.weekday == 0).ffill()
        df["hi_in"] = df["high"].rolling(self.ENTRY).max()     # entry lines
        df["lo_in"] = df["low"].rolling(self.ENTRY).min()
        df["hi_out"] = df["high"].rolling(self.EXIT).max()     # exit lines
        df["lo_out"] = df["low"].rolling(self.EXIT).min()
        df["hi_fs"] = df["high"].rolling(self.FAILSAFE).max()  # failsafe lines
        df["lo_fs"] = df["low"].rolling(self.FAILSAFE).min()
        return df

## 10. Part 3: the System 1 filter
Faith: skip a breakout if the **last** breakout — long or short, taken or skipped — would have won. It lost if the price moved 2N against it before a profitable 10-day exit. We replay that hypothetical trade candle by candle. Daily columns from section 9 arrive with the suffix `_1d`.

In [ ]:
%%writefile -a user_data/strategies/Turtle.py

    def populate_indicators(self, df, metadata):
        # s1_ok: may System 1 take a breakout on this candle?
        df["s1_ok"] = self.last_breakout_lost(df) if self.FILTER else True
        return df

    def last_breakout_lost(self, df):
        # plain arrays: a loop over 700,000 candles must be fast
        hi, lo = df["high"].to_numpy(), df["low"].to_numpy()
        n = df["N_1d"].to_numpy()
        hi_in, lo_in = df["hi_in_1d"].to_numpy(), df["lo_in_1d"].to_numpy()
        hi_out, lo_out = df["hi_out_1d"].to_numpy(), df["lo_out_1d"].to_numpy()
        ok = np.ones(len(df), dtype=bool)
        # no breakout seen yet -> the first one is allowed
        lost, side, entry, stop = True, 0, 0.0, 0.0
        for i in range(len(df)):
            ok[i] = lost                     # allowed only after a loser
            # flat: a breakout opens an imaginary trade, stop 2N away
            if side == 0 and hi[i] > hi_in[i]:
                side, entry, stop = 1, hi_in[i], hi_in[i] - 2 * n[i]
            elif side == 0 and lo[i] < lo_in[i]:
                side, entry, stop = -1, lo_in[i], lo_in[i] + 2 * n[i]
            # long: the 2N stop first -> it lost
            elif side == 1 and lo[i] <= stop:
                lost, side = True, 0
            # long: the 10-day exit first -> lost unless it is above the entry
            elif side == 1 and lo[i] < lo_out[i]:
                lost, side = lo_out[i] <= entry, 0
            # short: the same, mirrored
            elif side == -1 and hi[i] >= stop:
                lost, side = True, 0
            elif side == -1 and hi[i] > hi_out[i]:
                lost, side = hi_out[i] >= entry, 0
        return ok

## 11. Part 4: entries and exits, during the day
Enter when a 5-minute candle trades through the 20-day line and the filter allows it — or through the 55-day line, which is always taken (the failsafe). Exit all units when the price trades through the 10-day line. `s1_ok` from section 10, daily lines from 9.

In [ ]:
%%writefile -a user_data/strategies/Turtle.py

    def populate_entry_trend(self, df, metadata):
        # System 1: through the 20-day line, if the filter allows it
        s1_long = (df["high"] > df["hi_in_1d"]) & df["s1_ok"]
        s1_short = (df["low"] < df["lo_in_1d"]) & df["s1_ok"]
        # Faith's failsafe: the 55-day breakout is always taken
        df.loc[s1_long | (df["high"] > df["hi_fs_1d"]), "enter_long"] = 1
        df.loc[s1_short | (df["low"] < df["lo_fs_1d"]), "enter_short"] = 1
        return df

    def populate_exit_trend(self, df, metadata):
        # all units leave when the price breaks the 10-day line against us
        df.loc[df["low"] < df["lo_out_1d"], "exit_long"] = 1
        df.loc[df["high"] > df["hi_out_1d"], "exit_short"] = 1
        return df

## 12. Part 5: the notional account
Faith: the account size is set each January, and every 10% lost cuts it by 20% — 1,000,000 becomes 800,000, then 640,000 — until the year's starting equity is regained. Equity = the wallet plus open trades valued at the last close; `Trade.get_trades_proxy` lists the open trades.

In [ ]:
%%writefile -a user_data/strategies/Turtle.py

    def leverage(self, pair, **kwargs):
        return self.LEV

    def row(self, pair):
        # the newest analysed candle of a pair
        df, _ = self.dp.get_analyzed_dataframe(pair, self.timeframe)
        return df.iloc[-1]

    def equity(self):
        # wallet + what the open trades would make if closed now
        open_trades = Trade.get_trades_proxy(is_open=True)
        unrealised = sum(t.calc_profit(float(self.row(t.pair)["close"]))
                         for t in open_trades)
        return self.wallets.get_total_stake_amount() + unrealised

    def notional(self, current_time):
        eq = self.equity()
        # Faith: the account size is reset every January
        if getattr(self, "year", None) != current_time.year:
            self.year, self.base = current_time.year, eq
            self.size, self.floor = eq, 0.9 * eq
        # back at the year's start equity -> full size again
        if eq >= self.base:
            self.size, self.floor = self.base, 0.9 * self.base
        # every 10% lost -> trade as if the account were 20% smaller
        while eq <= self.floor:
            self.size *= 0.8
            self.floor -= 0.1 * self.size
        return self.size

## 13. Part 6: unit size and unit limits
Coins = 1% of the notional account / Monday's N; margin = coins × price / leverage. No new unit once 6 are open in one direction across BTC and ETH (Bitcoin and Ethereum). `notional`, `row` (12), `RISK`, `LEV`, `MAX_CORRELATED` (8).

In [ ]:
%%writefile -a user_data/strategies/Turtle.py

    def units_open(self, short):
        # every open unit has its own stop, so count the stops on one side
        trades = Trade.get_trades_proxy(is_open=True)
        return sum(len(t.get_custom_data("stops") or [])
                   for t in trades if t.is_short == short)

    def unit_margin(self, pair, rate, current_time):
        # Faith: 1 unit = 1% of the account / N   (section 6)
        n = float(self.row(pair)["N_sheet_1d"])
        coins = self.RISK * self.notional(current_time) / n
        return coins * rate / self.LEV          # freqtrade wants the margin

    def custom_stake_amount(self, pair, current_time, current_rate,
                            proposed_stake, min_stake, max_stake, leverage,
                            entry_tag, side, **kwargs):
        # Faith: at most 6 units in one direction across BTC and ETH
        if self.units_open(side == "short") >= self.MAX_CORRELATED:
            return 0
        margin = self.unit_margin(pair, current_rate, current_time)
        return min(margin, max_stake)

## 14. Part 7: a stop for every unit
Faith: each unit's stop is 2N from its fill, and every new unit raises the earlier stops by ½ N. Normally they all meet 2N below the latest unit; after a gap the latest unit's stop sits higher. `set_custom_data` keeps the list on the trade. `row` (12).

In [ ]:
%%writefile -a user_data/strategies/Turtle.py

    def order_filled(self, pair, trade, order, current_time, **kwargs):
        n = float(self.row(pair)["N_1d"])
        stops = trade.get_custom_data("stops") or []
        amounts = trade.get_custom_data("amounts") or []
        if order.ft_order_side == trade.entry_side:      # a unit was added
            away = 1 if trade.is_short else -1           # where a stop sits
            # Faith: earlier stops move 1/2 N up, the new one is 2N away
            stops = [s - away * 0.5 * n for s in stops]
            stops = stops + [order.safe_price + away * 2 * n]
            amounts = amounts + [order.safe_amount]
            trade.set_custom_data("N", n)               # N at this fill
            trade.set_custom_data("last", order.safe_price)  # for the next add
        else:                                            # newest unit sold
            stops, amounts = stops[:-1], amounts[:-1]
        trade.set_custom_data("stops", stops)            # kept on the trade
        trade.set_custom_data("amounts", amounts)

## 15. Part 8: adding units — and stopping out the newest one
Sell the newest unit alone if its own, higher stop is hit; otherwise add one unit each time the price moves ½ N past the last fill, up to 4 per market and 6 across the group. `N`, `last`, `stops`, `amounts` (14), `unit_margin`, `units_open` (13).

In [ ]:
%%writefile -a user_data/strategies/Turtle.py

    def adjust_trade_position(self, trade, current_time, current_rate,
                              current_profit, min_stake, max_stake,
                              current_entry_rate, current_exit_rate,
                              current_entry_profit, current_exit_profit,
                              **kwargs):
        if trade.has_open_orders:                 # one order at a time
            return None
        stops = trade.get_custom_data("stops")
        n, last = trade.get_custom_data("N"), trade.get_custom_data("last")
        # after a gap the newest unit's stop is tighter: it goes alone
        if len(stops) > 1:
            if trade.is_short:
                hit = stops[-1] < max(stops[:-1]) and current_rate >= stops[-1]
            else:
                hit = stops[-1] > min(stops[:-1]) and current_rate <= stops[-1]
            if hit:
                coins = trade.get_custom_data("amounts")[-1]
                return -coins * current_rate / trade.leverage, "unit_stop"
        # Faith: at most 4 units here, 6 across Bitcoin and Ethereum
        if len(stops) >= self.MAX_UNITS:
            return None
        if self.units_open(trade.is_short) >= self.MAX_CORRELATED:
            return None
        # Faith: add a unit every 1/2 N past the last fill
        if trade.is_short:
            moved = current_rate <= last - 0.5 * n
        else:
            moved = current_rate >= last + 0.5 * n
        if not moved:
            return None
        return min(self.unit_margin(trade.pair, current_rate, current_time),
                   max_stake)

## 16. Part 9: the position stop — and liquidation
The whole position goes at the earliest units' stop or at the liquidation price from episode q00 (funding included), whichever is closer. `stops` (14), `MMR` (8) — maintenance margin rate.

In [ ]:
%%writefile -a user_data/strategies/Turtle.py

    def custom_stoploss(self, pair, trade, current_time, current_rate,
                        current_profit, after_fill, **kwargs):
        stops = trade.get_custom_data("stops")
        # margin left after funding, as in episode q00
        m = 1 + (trade.funding_fees or 0) / trade.stake_amount
        if trade.is_short:
            liq = trade.open_rate * (1 + m / trade.leverage) / (1 + self.MMR)
            level = min(max(stops), liq)       # whichever comes first
        else:
            liq = trade.open_rate * (1 - m / trade.leverage) / (1 - self.MMR)
            level = max(min(stops), liq)
        return stoploss_from_absolute(level, current_rate,
                                      is_short=trade.is_short,
                                      leverage=trade.leverage)

## 17. Read freqtrade's verdict
`load_backtest_stats` returns freqtrade's own summary; `load_backtest_data` every trade. We print the figures that matter and the money funding and fees took.

In [ ]:
from freqtrade.data.btanalysis import load_backtest_data, load_backtest_stats

def newest():
    return max(Path("user_data/backtest_results").glob("*.zip"),
               key=lambda p: p.stat().st_mtime)

def verdict(name):
    s = load_backtest_stats(newest())["strategy"][name]   # freqtrade's summary
    t = load_backtest_data(newest())                      # every trade
    # fee = 0.05% of every order's value
    fees = sum(o["cost"] for orders in t["orders"] for o in orders) * 0.0005
    row = {"starting balance": s["starting_balance"],
           "final balance": s["final_balance"], "CAGR": s["cagr"],
           "max drawdown": s["max_drawdown_account"],
           "trades": s["total_trades"],
           "long profit": s["profit_total_long_abs"],
           "short profit": s["profit_total_short_abs"],
           "funding paid": t["funding_fees"].sum(), "fees paid": -fees}
    print(pd.Series(row).round(3).to_string())
    return row, t

## 18. Drawdown, marked to market
freqtrade's "max drawdown" counts only **closed** trades. Here every hour is valued at the market: cash from each fill plus the coins held × the hourly close; funding booked when a trade closes. Uses `DATA` (2).

In [ ]:
def equity(trades, account=10_000, start=None):
    # hourly closes of every coin traded
    prices = pd.DataFrame({
        p: pd.read_feather(DATA / f"{p.split('/')[0]}_USDT_USDT-1h-futures"
                                  ".feather").set_index("date")["close"]
        for p in trades["pair"].unique()}).ffill()
    rows = []
    for pair, short, orders in zip(trades["pair"], trades["is_short"],
                                   trades["orders"]):
        for o in orders:
            sign = (1 if o["ft_is_entry"] else -1) * (-1 if short else 1)
            when = pd.to_datetime(o["order_filled_timestamp"], unit="ms",
                                  utc=True)
            rows.append((when, pair, o["amount"] * sign, o["safe_price"],
                         o["cost"]))
    fills = pd.DataFrame(rows,
                         columns=["time", "pair", "coins", "price", "cost"])
    # cash moved by each fill, minus its fee
    fills["cash"] = -fills["coins"] * fills["price"] - fills["cost"] * 0.0005
    value = 0
    for pair in prices:
        held = fills[fills["pair"] == pair].groupby("time")[["coins", "cash"]]
        held = held.sum().cumsum()                     # running totals
        held = held.reindex(prices.index.union(held.index)).ffill()
        held = held.fillna(0).reindex(prices.index)    # on every hour
        value = value + held["cash"] + held["coins"] * prices[pair]
    paid = trades.groupby("close_date")["funding_fees"].sum().cumsum()
    paid = paid.reindex(prices.index.union(paid.index)).ffill()
    paid = paid.fillna(0).reindex(prices.index)        # booked at each close
    return (account + value + paid).loc[start:]

def drawdown(curve):
    # the deepest fall from the highest point reached so far
    return (1 - curve / curve.cummax()).max()

## 19. Run System 1
About 700,000 five-minute candles per coin — this takes several minutes. Uses `config.json` (7), `Turtle` (8–16), `verdict` (17), `equity`, `drawdown` (18).

In [ ]:
!freqtrade backtesting --cache none -c config.json -s Turtle --timerange 20200101-20261031 --export trades > run.log 2>&1
s1, trades = verdict("Turtle")
s1["drawdown, marked to market"] = drawdown(equity(trades))

## 20. Did every rule fire?
How the units ended (`unit_stop` = the newest unit's own stop), and how far from the breakout level we got filled, in each coin's own N (formula of section 5). Uses `trades` (19), `btc`, `eth` (3).

In [ ]:
sold = [o["ft_order_tag"] for orders in trades["orders"] for o in orders
        if not o["ft_is_entry"]]
print(pd.Series(sold).value_counts().to_string())       # how units ended
daily = {"BTC/USDT:USDT": btc, "ETH/USDT:USDT": eth}

def n_of(d):                                            # N, as in section 5
    prev = d["close"].shift()
    tr = pd.concat([d["high"] - d["low"], (d["high"] - prev).abs(),
                    (d["low"] - prev).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / 20, adjust=False).mean()

slip = []
for _, t in trades.iterrows():
    fill = t["orders"][0]                               # the first unit
    day = pd.to_datetime(fill["order_filled_timestamp"], unit="ms", utc=True)
    before = daily[t["pair"]].loc[:day.floor("D") - pd.Timedelta(days=1)]
    if t["is_short"]:
        level = before["low"].iloc[-20:].min()          # the line it broke
    else:
        level = before["high"].iloc[-20:].max()
    slip.append(abs(fill["safe_price"] - level) / n_of(before).iloc[-1])
slip = pd.Series(slip)
print("fill vs breakout, in N: median", round(slip.median(), 3),
      " 90% within", round(slip.quantile(0.9), 3))

## 21. The best trade, order by order
Uses `trades` (19).

In [ ]:
best = trades.loc[trades["profit_abs"].idxmax()]
print(best["pair"], "profit", round(best["profit_abs"]), "USDT")
for o in best["orders"]:
    print(pd.to_datetime(o["order_filled_timestamp"], unit="ms"),
          o["ft_order_side"], o["amount"], "at", o["safe_price"],
          o["ft_order_tag"])

## 22. Against simply holding
Buy-and-hold = half the account in BTC (Bitcoin), half in ETH (Ethereum), valued every hour the same way. Uses `equity`, `drawdown` (18), `trades` (19), `account` (6), `DATA` (2).

In [ ]:
hourly = {c: pd.read_feather(DATA / f"{c}_USDT_USDT-1h-futures.feather")
             .set_index("date")["close"] for c in ["BTC", "ETH"]}
btc_part = hourly["BTC"] / hourly["BTC"].iloc[0]
eth_part = hourly["ETH"] / hourly["ETH"].iloc[0]
hold = account * (btc_part + eth_part) / 2          # half in each coin
turtle = equity(trades)
for name, curve in [("buy and hold", hold), ("Turtle System 1", turtle)]:
    print(f"{name:>16}: final {curve.iloc[-1]:>9.0f}"
          f"  max drawdown {drawdown(curve):.0%}")
plt.figure(figsize=(11, 5))
plt.plot(hold.index, hold, label="buy and hold BTC+ETH")
plt.plot(turtle.index, turtle, label="Turtle System 1")
plt.yscale("log"); plt.legend(); plt.title("10,000 USDT, 2020 to 2026")
plt.show()

## 23. What the filter is worth
The same rules with the System 1 filter switched off — every 20-day breakout taken. Inherits sections 8–16.

In [ ]:
%%writefile user_data/strategies/TurtleNoFilter.py
from Turtle import Turtle

class TurtleNoFilter(Turtle):
    FILTER = False

### Run it without the filter
Uses `config.json` (7), `TurtleNoFilter` (above), `verdict` (17), `equity`, `drawdown` (18).

In [ ]:
!freqtrade backtesting --cache none -c config.json -s TurtleNoFilter --timerange 20200101-20261031 --export trades > run.log 2>&1
nofilter, t = verdict("TurtleNoFilter")
nofilter["drawdown, marked to market"] = drawdown(equity(t))

## 24. System 2: the slow version
55-day entry, 20-day exit, every breakout taken (Faith: System 2 has no filter). Inherits sections 8–16.

In [ ]:
%%writefile user_data/strategies/TurtleS2.py
from Turtle import Turtle

class TurtleS2(Turtle):
    ENTRY, EXIT, FILTER = 55, 20, False

### Run System 2
Uses `config.json` (7), `TurtleS2` (above), `verdict` (17), `equity`, `drawdown` (18).

In [ ]:
!freqtrade backtesting --cache none -c config.json -s TurtleS2 --timerange 20200101-20261031 --export trades > run.log 2>&1
s2, t = verdict("TurtleS2")
s2["drawdown, marked to market"] = drawdown(equity(t))

## 25. More leverage, same bet
The unit is sized in coins (section 13), so leverage changes only the margin — and how close liquidation sits: at 20x about 5% away, closer than 2N. Inherits sections 8–16.

In [ ]:
%%writefile user_data/strategies/Turtle20x.py
from Turtle import Turtle

class Turtle20x(Turtle):
    LEV = 20

### Run it at 20x
Uses `config.json` (7), `Turtle20x` (above), `verdict` (17), `equity`, `drawdown` (18).

In [ ]:
!freqtrade backtesting --cache none -c config.json -s Turtle20x --timerange 20200101-20261031 --export trades > run.log 2>&1
s20, t = verdict("Turtle20x")
s20["drawdown, marked to market"] = drawdown(equity(t))
print(t["exit_reason"].value_counts().to_string())

## 26. Does it matter how fast we enter?
The same System 1, watching 15-minute candles instead of 5-minute ones — up to 15 minutes later after a breakout. `--timeframe 15m` overrides the setting in `config.json` (7). Uses `verdict` (17), `equity`, `drawdown` (18).

In [ ]:
!freqtrade backtesting --cache none -c config.json -s Turtle --timeframe 15m --timerange 20200101-20261031 --export trades > run.log 2>&1
s15, t = verdict("Turtle")
s15["drawdown, marked to market"] = drawdown(equity(t))

## 27. Every run side by side
S1 = System 1, S2 = System 2. Uses `s1` (19), `nofilter` (23), `s2` (24), `s20` (25), `s15` (26).

In [ ]:
table = pd.DataFrame({"S1": s1, "no filter": nofilter, "S2": s2, "20x": s20,
                      "15 min": s15})
table.index = ["start", "final", "CAGR", "drawdown (freqtrade)", "trades",
               "long profit", "short profit", "funding", "fees",
               "drawdown (hourly)"]
print(table.to_string(float_format=lambda v: f"{v:,.0f}" if abs(v) >= 10
                      else f"{v:.2f}"))